In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model
model=init_chat_model("groq:openai/gpt-oss-120b", groq_api_key= os.getenv("GROQ_API_KEY"))
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title: str= Field(description="movie name"),
    year: int= Field(description="which year it released on"),
    genre: str= Field(description= "which genre")
model_with_structure=model.with_structured_output(Movie)
model_with_structure
model.invoke("provide short details of ramayanam")
model_with_structure.invoke("provide movie details of bahubali")
#print(response)


c:\Users\venkat vineeth\.venv\Lib\site-packages\pydantic\json_schema.py:2672: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='movie name'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
c:\Users\venkat vineeth\.venv\Lib\site-packages\pydantic\json_schema.py:2672: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='which year it released on'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


Movie(title='Baahubali: The Beginning', year=2015, genre='action')

In [3]:
from typing_extensions import TypedDict, Annotated
class MovieDict(TypedDict):
    """movie details"""
    movie_name: Annotated[str,...,"title of movie"]
    year: Annotated[int,...,"year released"]
model_with_typedict= model.with_structured_output(MovieDict)
response= model_with_typedict.invoke("provide movie details ramayana part 1")
response

{'movie_name': 'Ramayana Part 1', 'year': 2023}

In [10]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model
model=init_chat_model("groq:openai/gpt-oss-120b", groq_api_key= os.getenv("GROQ_API_KEY"),max_tokens=150,)
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

agent= create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)
config={"configurable":{"thread_id":"test1"}}
questions=[
    "what 2+2",
    "what 3+4",
    "what 4+5",
    "what 5+6",
    "what 6+7",
    "what 7+8"
]
for q in questions:
    response= agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(response)
    print(len(response['messages']))




{'messages': [HumanMessage(content='what 2+2', additional_kwargs={}, response_metadata={}, id='9193a0af-d236-47cc-8edd-e0599fad2c24'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'The user asks "what 2+2". Simple math. Answer 4.'}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 76, 'total_tokens': 113, 'completion_time': 0.076956469, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.03178248, 'prompt_tokens_details': None, 'queue_time': 0.318926197, 'total_time': 0.108738949}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8ea50c0161', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1e373-72a0-7280-a0eb-a2af02b60560-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 37, 'total_tokens': 113, 'output_token_details': {'reasoning': 18}})]}
2
{'messages': [HumanMessage

In [15]:
import os
from dotenv import load_dotenv
load_dotenv()
from groq import Groq

client = Groq()  # reads GROQ_API_KEY from the environment
structured = """You sort customer support tickets for an online shop.

Categories:
- billing: payments, charges, refunds
- shipping: parcels, delivery, damaged boxes
- other: anything else

Reply with one line of JSON and nothing else, like this:
{"category": "shipping", "priority": 3}
priority is 1 (can wait) to 5 (urgent)."""

client = Groq()  # reads GROQ_API_KEY from the environment
system = {"role": "system", "content": "You are FinCoach, a personal finance assistant. Reply in one short sentence."}
turns = [
    "My monthly take-home is ₹1,20,000.",
    "My expenses are ₹60,000 a month.",
    "What is the most important thing I should do this month?",
]
history = [system]
for turn in turns:
    history.append({"role": "user", "content": turn})
    response = client.chat.completions.create(model="openai/gpt-oss-120b", messages=history)
    history.append({"role": "assistant", "content": response.choices[0].message.content})
    print(len(history), "messages,", response.usage.prompt_tokens, "tokens sent")
conversation = [{"role": "system", "content": structured}]
for turn in range(40):
    conversation.append({"role": "user", "content": f"Ticket {turn}: my parcel has not arrived yet"})
    conversation.append({"role": "assistant", "content": '{"category": "shipping", "priority": 3}'})
import tiktoken

encoding = tiktoken.get_encoding("o200k_harmony")


def count_tokens(messages):
    return sum(len(encoding.encode(message["content"])) for message in messages)
def trim(messages, budget):
    system, rest = messages[:1], messages[1:]
    while rest and count_tokens(system + rest) > budget:
        rest = rest[2:]  # drop the oldest question and its answer together
    return system + rest
print(len(conversation), "messages,", count_tokens(conversation), "tokens")

short = trim(conversation, budget=200)
print(len(short), "messages,", count_tokens(short), "tokens")
print("oldest kept:", short[1]["content"])

3 messages, 102 tokens sent
5 messages, 143 tokens sent
7 messages, 186 tokens sent
81 messages, 956 tokens
11 messages, 186 tokens
oldest kept: Ticket 35: my parcel has not arrived yet


In [42]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
from langchain.agents  import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

model=init_chat_model("groq:openai/gpt-oss-120b", groq_api_key= os.getenv("GROQ_API_KEY"),max_tokens=150,)

def read_email_tool(emailid:str)->str:
    """mock function to read an email by its id"""
    return f"mail content for id:{emailid}"
def send_mail_tool(recipient:str,subject:str,body:str)->str:
    """mock function to send an mail"""
    return f"email sent to {recipient} with subject '{subject}'"
agent= create_agent(
    model= "groq:openai/gpt-oss-120b",
    tools=[read_email_tool,send_mail_tool],
    checkpointer= InMemorySaver(),
    middleware= [
        HumanInTheLoopMiddleware(
            interupt_on={
                "send_mail_tool":{
                    "allowed_decisions":[{"approve","edit","reject"}]
                },
                "read_email_tool":False
            }
        )
    ]
)
config= {"configurable":{"thread_id":"test_2"}}
result=agent.invoke({
    "messages":[HumanMessage(content="send mail to vijayakil13@gmail.com with subject 'Hello' and body 'How are you?'")]
},config=config)
result


TypeError: HumanInTheLoopMiddleware.__init__() got an unexpected keyword argument 'interupt_on'. Did you mean 'interrupt_on'?